<a href="https://colab.research.google.com/github/jaimeisaac2020/Python-analsisis-basicos/blob/mi-github/pronosticos_GA_julio_2025_tesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install pandas numpy tensorflow scikit-learn

In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import os

# --- 1. Carga de Datos desde el archivo CSV ---
# Nombre del archivo que debe estar en el mismo directorio que el script.
file_name = 'precios_cierre_2025_google_amazon.csv'

# Verificar si el archivo existe antes de continuar
if not os.path.exists(file_name):
    print(f"Error: El archivo '{file_name}' no se encontró.")
    print("Por favor, asegúrate de que el archivo CSV esté en la misma carpeta que el script.")
else:
    # Cargar los datos desde el archivo CSV
    df = pd.read_csv(file_name)
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)

    # Función para crear secuencias de datos para la LSTM
    def create_sequences(data, look_back=1):
        X, y = [], []
        for i in range(len(data) - look_back):
            X.append(data[i:(i + look_back), 0])
            y.append(data[i + look_back, 0])
        return np.array(X), np.array(y)

    # Función principal para entrenar y pronosticar
    def train_and_forecast(data, look_back, forecast_days):
        """
        Entrena un modelo LSTM y pronostica los precios futuros.

        :param data: Serie de precios históricos (una sola columna).
        :param look_back: Número de días pasados a usar para la predicción.
        :param forecast_days: Número de días a pronosticar en el futuro.
        :return: Array con los precios pronosticados.
        """
        # Escalar los datos
        scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_data = scaler.fit_transform(data.values.reshape(-1, 1))

        # Crear secuencias de entrenamiento
        X_train, y_train = create_sequences(scaled_data, look_back)
        X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

        # --- Construcción y Entrenamiento del Modelo LSTM ---
        model = Sequential([
            LSTM(50, return_sequences=True, input_shape=(look_back, 1)),
            Dropout(0.2),
            LSTM(50, return_sequences=False),
            Dropout(0.2),
            Dense(25),
            Dense(1)
        ])

        model.compile(optimizer='adam', loss='mean_squared_error')

        # El entrenamiento se hace con pocas 'epochs' para que sea rápido.
        # Para mayor precisión, se podrían aumentar.
        model.fit(X_train, y_train, batch_size=1, epochs=20, verbose=0)

        # --- Generación de Pronósticos ---
        last_sequence = scaled_data[-look_back:]
        current_batch = last_sequence.reshape((1, look_back, 1))
        future_predictions = []

        for i in range(forecast_days):
            current_pred = model.predict(current_batch, verbose=0)[0]
            future_predictions.append(current_pred)
            current_batch = np.append(current_batch[:, 1:, :], [[current_pred]], axis=1)

        # Invertir la escala de las predicciones
        forecasted_prices = scaler.inverse_transform(future_predictions)

        return forecasted_prices.flatten()

    # --- Ejecución del Pronóstico para cada Acción ---

    # Definir parámetros
    LOOK_BACK = 60
    LAST_DATE = df.index[-1]

    # Calcular los días a pronosticar (días hábiles hasta fin de julio 2025)
    forecast_dates = pd.bdate_range(start=LAST_DATE + pd.Timedelta(days=1), end='2025-07-31')
    FORECAST_DAYS = len(forecast_dates)

    print("Iniciando el pronóstico...")
    print(f"Leyendo datos desde: {file_name}")
    print(f"Datos históricos hasta: {LAST_DATE.date()}")
    print(f"Período a pronosticar: {forecast_dates[0].date()} a {forecast_dates[-1].date()} ({FORECAST_DAYS} días hábiles)")
    print("-" * 50)

    # Pronóstico para Amazon (AMZN)
    print("Entrenando modelo y pronosticando para Amazon (AMZN)...")
    amzn_data = df[['AMZN_Close']]
    amzn_forecast = train_and_forecast(amzn_data, LOOK_BACK, FORECAST_DAYS)
    print("Pronóstico para AMZN completado.")
    print("-" * 50)

    # Pronóstico para Google (GOOGL)
    print("Entrenando modelo y pronosticando para Google (GOOGL)...")
    googl_data = df[['GOOGL_Close']]
    googl_forecast = train_and_forecast(googl_data, LOOK_BACK, FORECAST_DAYS)
    print("Pronóstico para GOOGL completado.")
    print("-" * 50)

    # --- Presentación de Resultados ---

    # Crear un DataFrame con los resultados
    forecast_df = pd.DataFrame({
        'Date': forecast_dates,
        'AMZN_Forecast': amzn_forecast,
        'GOOGL_Forecast': googl_forecast
    })

    # Formatear la salida para una mejor lectura
    forecast_df['AMZN_Forecast'] = forecast_df['AMZN_Forecast'].map('${:,.2f}'.format)
    forecast_df['GOOGL_Forecast'] = forecast_df['GOOGL_Forecast'].map('${:,.2f}'.format)
    forecast_df.set_index('Date', inplace=True)

    print("\n*** PRONÓSTICO DE PRECIOS DE CIERRE (JUNIO - JULIO 2025) ***\n")
    print(forecast_df)

Iniciando el pronóstico...
Leyendo datos desde: precios_cierre_2025_google_amazon.csv
Datos históricos hasta: 2025-06-27
Período a pronosticar: 2025-06-30 a 2025-07-31 (24 días hábiles)
--------------------------------------------------
Entrenando modelo y pronosticando para Amazon (AMZN)...


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Pronóstico para AMZN completado.
--------------------------------------------------
Entrenando modelo y pronosticando para Google (GOOGL)...


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Pronóstico para GOOGL completado.
--------------------------------------------------

*** PRONÓSTICO DE PRECIOS DE CIERRE (JUNIO - JULIO 2025) ***

           AMZN_Forecast GOOGL_Forecast
Date                                   
2025-06-30       $207.67        $169.46
2025-07-01       $207.26        $169.36
2025-07-02       $206.18        $168.96
2025-07-03       $204.81        $168.43
2025-07-04       $203.30        $167.85
2025-07-07       $201.73        $167.25
2025-07-08       $200.14        $166.65
2025-07-09       $198.53        $166.05
2025-07-10       $196.93        $165.46
2025-07-11       $195.34        $164.88
2025-07-14       $193.77        $164.31
2025-07-15       $192.23        $163.75
2025-07-16       $190.73        $163.21
2025-07-17       $189.27        $162.68
2025-07-18       $187.88        $162.18
2025-07-21       $186.56        $161.70
2025-07-22       $185.32        $161.24
2025-07-23       $184.16        $160.82
2025-07-24       $183.09        $160.42
2025-07-25  